# Maneuver Classification — Random Forest

Replace the heuristic `LaneClassifier + TurnIndicator` pair with a data-driven
**Random Forest** trained on sliding-window features from the IMU signal.

| Step | What happens |
|---|---|
| Feature extraction | 39 stats per 1-second window |
| Labels | Auto-generated from trip segment schedule |
| Model | Random Forest (300 trees, balanced class weights) |
| Evaluation | 5-fold stratified CV + confusion matrix |
| Comparison | Side-by-side with rule-based classifier |

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent.parent   # project root
sys.path.insert(0, str(ROOT / 'ml'))
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from scooter_imu.data.loader       import load_raw
from scooter_imu.data.preprocessor import preprocess
from scooter_imu.algorithms.lane_classifier import LaneClassifier

from features    import extract_windows, feature_names
from labeler     import label_timestamps, CLASS_NAMES, CLASS_TO_IDX
from classifiers import ManeuverClassifier

df = preprocess(load_raw(ROOT / 'data/raw/ScooterIMUData.csv'))
print(f'Loaded {len(df):,} rows  |  duration: {df["timestamp"].iloc[-1]-df["timestamp"].iloc[0]}')

Loaded 30,000 rows  |  duration: 0 days 00:09:59.980000


## 1. Feature Extraction & Label Generation

In [2]:
X, mid_ts = extract_windows(df, window_s=1.0, step_s=0.5)
y_int     = label_timestamps(mid_ts, df['timestamp'].iloc[0])

# Drop unlabelled windows
mask  = y_int >= 0
X, y  = X[mask], y_int[mask]
y_str = np.array(CLASS_NAMES)[y]

print(f'Feature matrix: {X.shape}  (windows × features)')
print(f'Feature names (39 total):\n  {feature_names()}')

# Label distribution
unique, counts = np.unique(y_str, return_counts=True)
fig = go.Figure(go.Bar(x=list(unique), y=list(counts),
                       marker_color='#2E86AB',
                       text=list(counts), textposition='outside'))
fig.update_layout(title='Window Count per Maneuver Class',
                  xaxis_title='Class', yaxis_title='Windows',
                  height=350, template='plotly_white')
fig.show()

Feature matrix: (1199, 39)  (windows × features)
Feature names (39 total):
  ['gyrx_mean', 'gyrx_std', 'gyrx_min', 'gyrx_max', 'gyrx_rms', 'gyrx_zcr', 'gyrx_fft1', 'gyrx_fft2', 'gyrx_fft3', 'gyry_mean', 'gyry_std', 'gyry_min', 'gyry_max', 'gyry_rms', 'gyry_zcr', 'gyry_fft1', 'gyry_fft2', 'gyry_fft3', 'gyrz_mean', 'gyrz_std', 'gyrz_min', 'gyrz_max', 'gyrz_rms', 'gyrz_zcr', 'gyrz_fft1', 'gyrz_fft2', 'gyrz_fft3', 'mag_mean', 'mag_std', 'mag_min', 'mag_max', 'mag_rms', 'mag_zcr', 'mag_fft1', 'mag_fft2', 'mag_fft3', 'corr_xy', 'corr_yz', 'corr_xz']


## 2. Train / Test Split & Cross-Validation

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_tr)}  |  Test: {len(X_te)}')

mc = ManeuverClassifier(n_estimators=300)
cv_scores = mc.cross_validate(X_tr, y_tr, cv=5)
print(f'\n5-fold CV accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.3f}')

mc.fit(X_tr, y_tr, CLASS_NAMES)
test_acc = accuracy_score(y_te, mc.clf.predict(X_te))
print(f'\nTest accuracy: {test_acc:.3f}')

Train: 959  |  Test: 240



5-fold CV accuracy: 0.974 ± 0.009
  Fold 1: 0.979
  Fold 2: 0.958
  Fold 3: 0.984
  Fold 4: 0.974
  Fold 5: 0.974



Test accuracy: 0.971


## 3. Confusion Matrix

In [4]:
cm = mc.confusion(X_te, y_te)
present_classes = [CLASS_NAMES[i] for i in sorted(np.unique(y_te))]

fig = ff.create_annotated_heatmap(
    z=cm.tolist(),
    x=present_classes,
    y=present_classes,
    colorscale='Blues',
    showscale=True,
)
fig.update_layout(
    title='Confusion Matrix — Test Set',
    xaxis_title='Predicted', yaxis_title='Actual',
    height=500, template='plotly_white',
)
fig.show()
print(mc.report(X_te, y_te))

              precision    recall  f1-score   support

      parked       0.89      0.97      0.93        32
    straight       0.98      1.00      0.99       136
   left_turn       1.00      0.92      0.96        12
  right_turn       1.00      1.00      1.00        12
 lane_change       1.00      0.92      0.96        12
  sharp_turn       1.00      1.00      1.00         8
        fall       0.00      0.00      0.00         4
  bumpy_road       1.00      1.00      1.00        12
  aggressive       1.00      1.00      1.00        12

    accuracy                           0.97       240
   macro avg       0.87      0.87      0.87       240
weighted avg       0.96      0.97      0.96       240



## 4. Feature Importance

In [5]:
names = feature_names()
imp   = mc.feature_importances_
order = np.argsort(imp)[::-1][:20]

fig = go.Figure(go.Bar(
    x=[names[i] for i in order],
    y=[imp[i]   for i in order],
    marker_color='#A23B72',
))
fig.update_layout(
    title='Top 20 Feature Importances',
    xaxis_tickangle=-45,
    yaxis_title='Importance',
    height=420, template='plotly_white',
)
fig.show()

## 5. Compare ML vs Rule-Based Classifier

In [6]:
# Map rule-based maneuver labels to the same label set
RULE_MAP = {
    'straight':   'straight',
    'lane_change':'lane_change',
    'turn':       'left_turn',   # not direction-aware
    'sharp_turn': 'sharp_turn',
}

rb = LaneClassifier().predict(df)
rb_mapped = rb['maneuver'].map(RULE_MAP).fillna('straight')

# Ground-truth at sample level
gt_int = label_timestamps(df['timestamp'].to_numpy(), df['timestamp'].iloc[0])
gt_str = np.array(['unknown'] + CLASS_NAMES)[gt_int + 1]  # shift for 'unknown' at -1
gt_str = pd.array(gt_str)

# ML: predict for the same samples using window centre mapping
# Build per-sample ML labels by propagating window predictions
X_all, ts_all = extract_windows(df, window_s=1.0, step_s=0.02)
y_all_ml = mc.predict(X_all)

print('Rule-based maneuver distribution:')
print(rb['maneuver'].value_counts().to_string())
print('\nML classifier distribution:')
print(pd.Series(y_all_ml).value_counts().to_string())

Rule-based maneuver distribution:
maneuver
straight       24356
turn            4296
lane_change      935
sharp_turn       413

ML classifier distribution:
straight       17069
parked          4238
right_turn      1529
aggressive      1519
bumpy_road      1511
lane_change     1473
left_turn       1432
sharp_turn      1007
fall             173


## 6. Save Model

In [7]:
mc.save(ROOT / 'ml/models/maneuver_rf.joblib')
print('Model saved. Reload with:  ManeuverClassifier.load()')

ManeuverClassifier saved → /Users/pulkit/Desktop/test/Automatic-Indicator-using-Gyroscope-data/ml/models/maneuver_rf.joblib
Model saved. Reload with:  ManeuverClassifier.load()
